# Security size × fund size — nine-block study

Every calculation is run separately in the nine `security_size × fund_size` blocks. One panel row is one **security–quarter–fund-size** observation. Market variables belong to the security; active weights, holding changes, and buying-pressure targets are recomputed using only funds in that fund-size bucket. Models, target ranks, prediction quintiles, and scores never mix blocks.

Within a fund-size block, `fund_turnover` uses its matching source column: small funds use `security_small_fund_turnover`, mid funds use `security_mid_fund_turnover`, and large funds use `security_large_fund_turnover`.

## Targets

All targets describe **q → q+1** and are therefore outcomes, not contemporaneous features.

| target | concise definition | larger value means |
|---|---|---|
| `turnover_next` | the security's market turnover in quarter q+1 | more secondary-market trading |
| `ret_next` | the security's return from q to q+1 | higher next-quarter return |
| `buy_frac` | fraction of funds in this fund-size block whose shares rise by at least 1% | broader buying participation |
| `dollar_buy_frac` | the same buy indicator, weighted by each fund's position value | more invested dollars belong to buyers |
| `flow_pct_cap` | sum of `chg_pct × position_value`, divided by market cap | stronger signed net buying |
| `weight_chg` | mean `weight(q+1) − weight(q)` across funds in the block | larger portfolio-weight increases |
| `active_weight_chg` | mean weight change after removing passive price drift | stronger active reallocation into the security |
| `buy_weight_ratio` | positive weight changes divided by total absolute weight changes | more of weight reallocation is buying; 0.5 is balanced |
| `buy_dollar_ratio` | positive dollar changes divided by total absolute dollar changes | more of dollar reallocation is buying; 0.5 is balanced |

## Timing and interpretation

`security_size` and `fund_size` are keys known at q, not features. Forward holder decisions enter the model only through exact one-quarter lags. The three fund-turnover columns are assumed to describe q−1 → q; set `assume_fund_turnover_backward=False` if their true window is q → q+1.

`rank_IC` measures whether the target itself is predictable. `Q5_Q1_per_q` always measures a **return** spread: securities with the highest predicted target minus those with the lowest predicted target, sorted inside the same block and quarter.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
MODULE_DIR = ROOT / "mimicking_pipeline" if (ROOT / "mimicking_pipeline").is_dir() else ROOT
ROOT = MODULE_DIR.parent
sys.path.insert(0, str(MODULE_DIR))

import size_block_study as S
S.check_version()
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)


## 1. Configuration

`align_block_folds=True` forces all non-thin blocks to use the same test quarters. That makes every 3×3 comparison a same-calendar comparison.


In [ ]:
CFG = S.Config(
    holdings_path = str(ROOT / "manager_holdings/master_batches_return_filtered/master_all_funds_add_filter_ivy_rank_active_rank.parquet"),
    inv_type_codes = (401,),

    # False if security_*_fund_turnover describes q -> q+1 instead of q-1 -> q
    assume_fund_turnover_backward = True,

    security_strata = (0, 1, 2),    # small, mid, large securities
    fund_strata = (0, 1, 2),        # small, mid, large funds
    min_block_rows = 2000,

    min_quarters = 8,
    window_q = 28, test_q = 8, step = 8,
    align_eval_sample = True,
    align_block_folds = True,

    model = "hgb",               # "hgb" or "linear"
    train_target_transform = "rank",
)
CFG


## 2. Build the nine-block panel

Check the printed scale decisions, turnover convention, 3×3 row counts, and target coverage before fitting anything. A thin block can make the common-fold intersection empty or its estimates noisy.


In [ ]:
panel = S.build_panel(CFG)
print("\nfeatures :", S.feature_list(panel))
print("pressure :", S.pressure_list(panel))
display(S.block_counts(panel))


In [ ]:
# Every key must be unique: one security-quarter observation in each fund-size block.
assert not panel.duplicated(["security", "yq", "fund_size"]).any()

coverage_cols = [c for c in [
    "turnover_next", "ret_next", *[S.PRESSURE[k] for k in S.pressure_list(panel)]
] if c in panel.columns]
coverage = (panel.groupby(["security_label", "fund_label"])[coverage_cols]
            .agg(lambda x: x.notna().mean()).round(3))
display(coverage)


In [ ]:
# The two ratios should be bounded in [0, 1].  Their centre has the natural meaning 0.5.
ratio_cols = [c for c in ["buy_weight_ratio", "buy_dollar_ratio"] if c in panel]
if ratio_cols:
    display(panel.groupby(["security_label", "fund_label"])[ratio_cols]
            .agg(["count", "mean", "min", "max"]).round(4))
else:
    print("No ratio target could be built from the available columns.")


## 3. Turnover and returns in the 3×3 blocks

Each cell below is estimated from a separate rolling model and a separate within-block quintile sort.


In [ ]:
RETURN_TARGETS = ["turnover_next", "ret_next"]
TAB_RET = S.run_blocks(panel, CFG, targets=RETURN_TARGETS)
display(S.summary(TAB_RET))


In [ ]:
for target in RETURN_TARGETS:
    print(f"\n{target}: predictability, return spread, and spread t-stat")
    display(pd.concat({
        "rank_IC": S.block_matrix(TAB_RET, target, "rank_IC"),
        "Q5_Q1_per_q": S.block_matrix(TAB_RET, target, "Q5_Q1_per_q"),
        "spread_t": S.block_matrix(TAB_RET, target, "spread_t"),
    }, axis=1))


In [ ]:
print("edge > 0 means all features beat the best raw one-characteristic sort in |IC|.")
display(S.beats_naive(TAB_RET))


## 4. Buying pressure in the 3×3 blocks

This run fits all available buying-pressure definitions listed in the first cell. First read `rank_IC`: can the block's future trading be predicted? Then read the return spread: does that predicted trading carry alpha?


In [ ]:
PRESSURE_TARGETS = [S.PRESSURE[k] for k in S.pressure_list(panel)]
print(PRESSURE_TARGETS)
TAB_BP = S.run_blocks(panel, CFG, targets=PRESSURE_TARGETS)
display(S.summary(TAB_BP))


### 4a. Is each buying-pressure target predictable?


In [ ]:
for target in PRESSURE_TARGETS:
    print(f"\nrank_IC — {target}")
    display(S.block_matrix(TAB_BP, target, "rank_IC"))

edge = S.beats_naive(TAB_BP)
display(edge)


### 4b. Does predicted buying pressure carry next-quarter alpha?

These are return results even though the fitted target is buying pressure. A sign change across either axis is economically more informative than the average across all nine cells.


In [ ]:
for target in PRESSURE_TARGETS:
    print(f"\nnext-quarter return spread — predicted {target}")
    display(pd.concat({
        "Q5_Q1_per_q": S.block_matrix(TAB_BP, target, "Q5_Q1_per_q"),
        "spread_t": S.block_matrix(TAB_BP, target, "spread_t"),
    }, axis=1))


## 5. Reading the result

- **Move across columns:** for the same security size, does the result change when buying comes from small versus large funds?
- **Move down rows:** for the same fund size, does its signal work differently in small versus large securities?
- **`edge` near zero:** the model mainly learned persistence or one characteristic; extra features add little.
- **High `rank_IC`, weak return spread:** fund behavior is predictable but does not carry alpha.
- **Few test quarters or thin coverage:** treat t-statistics as indicative, not decisive.
- **Returns are gross:** trading costs can differ sharply across the nine blocks.

Before treating pressure results as final, confirm whether `chg_pct = -100%` means a true exit or missing next-quarter position. The default follows the existing study and excludes it from pressure measures.


## 6. Save


In [ ]:
OUT = MODULE_DIR / "outputs_size_blocks"
OUT.mkdir(exist_ok=True)
TAB_RET.to_csv(OUT / "size_block_returns.csv", index=False)
TAB_BP.to_csv(OUT / "size_block_buy_pressure.csv", index=False)
panel.to_parquet(OUT / "size_block_panel.parquet", index=False)
print(f"saved to {OUT}")
